# Refined Bad-Frame Trace HTML Viewer

Generate four pooled Plotly trace-viewer HTML files from the refined manual sidecar payloads. Bad-frame shading uses the same final per-cell bad mask as `Unified_CKII_Pipeline_refined.ipynb` (`snr_threshold=5`, `min_good_minutes=5.0`).

In [1]:
# Setup: paths, imports, and refined pipeline configuration.
import csv
import os
import pickle
import sys
from pathlib import Path

import numpy as np
import pandas as pd


def _find_project_root() -> Path:
    here = Path.cwd().resolve()
    candidates = [here, *here.parents]
    for candidate in candidates:
        if (candidate / 'utils' / 'placecell_pipeline.py').exists():
            return candidate
        nested = candidate / 'miniVI_PlaceCell_analysis_V4'
        if (nested / 'utils' / 'placecell_pipeline.py').exists():
            return nested
    raise RuntimeError('Could not locate miniVI_PlaceCell_analysis_V4 project root.')


project_root = _find_project_root()
repo_root = project_root.parent
data_root = project_root / 'data'
figures_root = project_root / 'figures'
notebooks_root = project_root / 'notebooks_PCs_refined'
output_dir = figures_root / 'CKII_pooled' / 'refined_trace_viewers'
output_dir.mkdir(parents=True, exist_ok=True)

# Ensure `utils` resolves to miniVI_PlaceCell_analysis_V4/utils, not any repo-level package.
sys.path = [p for p in sys.path if Path(p or Path.cwd()).resolve() != repo_root]
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import utils.placecell_pipeline as _pcp
from utils.placecell_pipeline import AnalysisParams, PipelineConfig, PlaceCellParams, PooledParams
from utils.spatial_heatmaps import cell_has_cs_place_field, is_csplus_place_cell

from dash_data_viewer_app.data_io import AnimalData, _as_per_cell_arrays, _as_per_cell_entries
import dash_data_viewer_app.figure_builder as fb

animals = [
    'CKII_pAce21_PR_20250806',
    'CKII_pAce38_PX_20251126',
    'CKII_pAce45_PX_20260118',
    'CKII_pAce47_PX_20260128',
    'CKII_pAce46_PR_20260222',
    'CKII_pAce50_PRL_20260317',
    'CKII_pAce54_PR_20260506',
    'CKII_pAce54_PX_20260514',
]

analysis_params = AnalysisParams(
    speed_threshold=3.0,
    speed_threshold_quiet=0.5,
    behavior_speed_outlier_threshold_cm_s=60.0,
    behavior_speed_outlier_cleaning=True,
    min_duration_s=0.25,
    merge_gap_s=0.0,
    kernel_size=51,
    snr_threshold=5.0,
    min_good_minutes=5.0,
    theta_freqs=(4.0, 8.0),
    slow_freqs=2.0,
    refined_apply_cb_baseline_removal=True,
    refined_cb_baseline_window_s=5.0,
    refined_snr_cb_baseline_window_s=1.0,
)

place_cell_params = PlaceCellParams(
    bin_size=1.5,
    place_field_threshold=0.35,
    min_component_peak_ratio=0.45,
    split_multi_peak_fields=False,
    split_secondary_peak_ratio=0.6,
    split_secondary_peak_min_separation_cm=6.0,
    min_peak_rate=0.5,
    max_field_area_ratio=0.5,
    min_field_bins=10,
    min_pf_firing_traversals=5,
    pf_firing_traversal_distance_window_cm=15.0,
    pf_firing_traversal_detection_window_cm=8.0,
    pf_firing_traversal_distance_bin_cm=1.5,
    pf_firing_traversal_distance_mode='euclidean_to_peak',
    pf_firing_traversal_center_vicinity_min_cm=1,
    pf_firing_traversal_center_vicinity_max_cm=5,
    pf_firing_traversal_resting_speed_threshold=0.5,
    pf_firing_traversal_merge_gap_s=2.0,
    pf_firing_traversal_exclude_trials_with_bad_frames=True,
    pf_reliability_dilation_bins=3,
    pf_reliability_dilation_shape='disk',
    smooth_sigma=1.5,
    min_occupancy_s=0.001,
    occ_smooth_sigma=1.5,
    num_shuffles=1000,
    random_seed=42,
    ss_shape_min_separation_ms=14.0,
    trim_sparse_top_row_for_analysis=True,
    trim_sparse_top_row_for_plotting=True,
    sparse_top_row_nonocc_frac_threshold=0.8,
)

pooled_params = PooledParams(
    cb_num_threshold=5,
    cs_peak_rate_threshold=0.5,
    cs_plc_definition_mode='cs_place_field',
)

config = PipelineConfig(
    project_root=project_root,
    data_root=data_root,
    figures_root=figures_root,
    notebooks_root=notebooks_root,
    animals=animals,
    merged_data_filename='manual_spike_detection_results.pkl',
    analysis=analysis_params,
    place_cell=place_cell_params,
    pooled=pooled_params,
)

SEGMENT_DURATION_S = 120.0
BAD_FILL_DARK = 'rgba(45, 45, 45, 0.70)'
print(f'Project root: {project_root}')
print(f'Output directory: {output_dir}')
print(f'SNR threshold: {config.analysis.snr_threshold:g}; min good minutes: {config.analysis.min_good_minutes:g}')

Project root: /Users/qixinyang/Documents/AdamLab/miniVI_Renana_Pipeline/miniVI_PlaceCell_analysis_V4
Output directory: /Users/qixinyang/Documents/AdamLab/miniVI_Renana_Pipeline/miniVI_PlaceCell_analysis_V4/figures/CKII_pooled/refined_trace_viewers
SNR threshold: 5; min good minutes: 5


## Load Refined Payloads and Final Bad Masks

Each animal is loaded from the refined analysis payload embedded in `manual_spike_detection_results.pkl`. The final bad mask is recomputed with `_compute_bad_masks`, matching the refined pooled pipeline.

In [2]:
def _get_cell_vector_from_source(source, cell_idx, n_frames, fill=np.nan):
    arr = None
    try:
        if isinstance(source, (list, tuple)) and int(cell_idx) < len(source):
            arr = np.asarray(source[int(cell_idx)], dtype=float).reshape(-1)
        else:
            data = np.asarray(source, dtype=float)
            if data.ndim == 2 and data.shape[0] > int(cell_idx) and data.shape[1] == int(n_frames):
                arr = data[int(cell_idx), :].reshape(-1)
            elif data.ndim == 2 and data.shape[1] > int(cell_idx) and data.shape[0] == int(n_frames):
                arr = data[:, int(cell_idx)].reshape(-1)
    except Exception:
        arr = None
    if arr is None:
        return np.full(int(n_frames), fill, dtype=float)
    if arr.size == int(n_frames):
        return arr
    out = np.full(int(n_frames), fill, dtype=float)
    n = min(int(n_frames), int(arr.size))
    if n > 0:
        out[:n] = arr[:n]
    return out


def _spatial_category_by_cell(animal_id):
    animal_dir = Path(config.data_root) / str(animal_id)
    spatial_path = animal_dir / 'spatial_analysis_full.pkl'
    if not spatial_path.exists():
        raise FileNotFoundError(f'Missing spatial analysis: {spatial_path}')
    with spatial_path.open('rb') as f:
        spatial_cells = pickle.load(f)
    out = {}
    for pos, cell in enumerate(spatial_cells if isinstance(spatial_cells, list) else []):
        if not isinstance(cell, dict):
            continue
        try:
            cell_idx = int(cell.get('cell_idx', pos))
        except Exception:
            continue
        is_place_cell = bool(cell.get('is_place_cell', False))
        n_cb = 0
        spike_shapes = cell.get('spike_shapes')
        if isinstance(spike_shapes, dict):
            complex_payload = spike_shapes.get('complex', {})
            if isinstance(complex_payload, dict):
                shapes = complex_payload.get('shapes', {})
                if isinstance(shapes, dict):
                    n_cb = int(len(shapes.get('run_in', [])))
        try:
            cs_peak_rate = float(cell.get('cs_peak_rate', np.nan))
        except Exception:
            cs_peak_rate = np.nan
        is_csplus = is_csplus_place_cell(
            is_place_cell=is_place_cell,
            n_cb_in_pf=int(n_cb),
            cs_peak_rate=cs_peak_rate,
            cb_num_threshold=int(config.pooled.cb_num_threshold),
            cs_peak_rate_threshold=float(config.pooled.cs_peak_rate_threshold),
            has_cs_place_field=cell_has_cs_place_field(cell),
            cs_plc_definition_mode=str(config.pooled.cs_plc_definition_mode),
        )
        category = 'CS+ PLC' if is_csplus else ('CS- PLC' if is_place_cell else 'Non-PLC')
        out[int(cell_idx)] = {
            'category': category,
            'is_place_cell': bool(is_place_cell),
            'is_csplus': bool(is_csplus),
            'spatial_entry': cell,
        }
    return out


refined_animals = []
cell_records = []
unclassified_eligible = []

for animal_id in config.animals:
    animal_dir = Path(config.data_root) / str(animal_id)
    merged = _pcp.load_manual_refined_analysis_data(animal_dir, sidecar_filename='manual_spike_detection_results.pkl')
    bad_masks, mask_stats = _pcp._compute_bad_masks(
        merged,
        float(config.analysis.snr_threshold),
        float(config.analysis.min_good_minutes),
        return_stats=True,
    )
    bad_masks = np.asarray(bad_masks, dtype=bool)
    frame_rate = float(merged.get('frame_rate', np.nan))
    x_neural = np.asarray(merged.get('x_neural', []), dtype=float).reshape(-1)
    y_neural = np.asarray(merged.get('y_neural', np.full_like(x_neural, np.nan)), dtype=float).reshape(-1)
    speed = np.asarray(merged.get('speed', np.full_like(x_neural, np.nan)), dtype=float).reshape(-1)
    n_frames = int(x_neural.size)
    n_cells = int(len(merged.get('spikes', [])))
    if bad_masks.shape != (n_cells, n_frames):
        raise RuntimeError(f'{animal_id}: bad mask shape mismatch: {bad_masks.shape} vs {(n_cells, n_frames)}')

    category_by_idx = _spatial_category_by_cell(animal_id)
    place_cell_mask = np.zeros(n_cells, dtype=bool)
    csplus_plc_mask = np.zeros(n_cells, dtype=bool)
    plc_categories = ['Unclassified' for _ in range(n_cells)]
    cb_in_pf_counts = np.zeros(n_cells, dtype=int)
    cs_peak_rates = np.full(n_cells, np.nan, dtype=float)

    for cell_idx in range(n_cells):
        bad_mask = bad_masks[cell_idx]
        is_excluded = bool(np.all(bad_mask))
        included_frames = int(np.sum(~bad_mask))
        included_minutes = float(included_frames) / frame_rate / 60.0 if np.isfinite(frame_rate) and frame_rate > 0 else np.nan
        pct_bad = float(100.0 * np.sum(bad_mask) / n_frames) if n_frames > 0 else np.nan

        spatial_meta = category_by_idx.get(int(cell_idx))
        if is_excluded:
            category = 'Excluded'
            is_place_cell = False
            is_csplus = False
        elif isinstance(spatial_meta, dict):
            category = str(spatial_meta['category'])
            is_place_cell = bool(spatial_meta['is_place_cell'])
            is_csplus = bool(spatial_meta['is_csplus'])
            entry = spatial_meta.get('spatial_entry', {})
            spike_shapes = entry.get('spike_shapes') if isinstance(entry, dict) else None
            if isinstance(spike_shapes, dict):
                complex_payload = spike_shapes.get('complex', {})
                if isinstance(complex_payload, dict):
                    shapes = complex_payload.get('shapes', {})
                    if isinstance(shapes, dict):
                        cb_in_pf_counts[cell_idx] = int(len(shapes.get('run_in', [])))
            try:
                cs_peak_rates[cell_idx] = float(entry.get('cs_peak_rate', np.nan)) if isinstance(entry, dict) else np.nan
            except Exception:
                cs_peak_rates[cell_idx] = np.nan
        else:
            category = 'Unclassified'
            is_place_cell = False
            is_csplus = False
            unclassified_eligible.append((animal_id, int(cell_idx)))

        place_cell_mask[cell_idx] = bool(is_place_cell)
        csplus_plc_mask[cell_idx] = bool(is_csplus)
        plc_categories[cell_idx] = category
        cell_records.append({
            'animal_id': animal_id,
            'cell_idx': int(cell_idx),
            'cell_number': int(cell_idx) + 1,
            'category': category,
            'is_excluded': bool(is_excluded),
            'included_frames': int(included_frames),
            'included_minutes': included_minutes,
            'n_frames': int(n_frames),
            'pct_bad_frames': pct_bad,
            'snr_threshold': float(config.analysis.snr_threshold),
            'min_good_minutes': float(config.analysis.min_good_minutes),
        })

    traces = merged.get('traces_SNR_interpolated', merged.get('traces', []))
    vms = merged.get('Vm_SNR_interpolated', traces)
    session_start_frames = [int(v) for v in np.asarray(merged.get('session_start_frames', [0]), dtype=int).reshape(-1)]
    if not session_start_frames:
        session_start_frames = [0]
    pos_nan_mask = (~np.isfinite(x_neural)) | (~np.isfinite(y_neural)) | (~np.isfinite(speed))

    merged_for_viewer = dict(merged)
    merged_for_viewer['_refined_bad_masks'] = bad_masks
    merged_for_viewer['_refined_mask_stats'] = list(mask_stats)

    refined_animals.append(AnimalData(
        animal_id=str(animal_id),
        path=animal_dir / 'manual_spike_detection_results.pkl',
        merged=merged_for_viewer,
        frame_rate=frame_rate,
        n_frames=n_frames,
        n_cells=n_cells,
        traces=traces,
        vms=vms,
        all_spikes=_as_per_cell_arrays(merged.get('all_spikes', merged.get('spikes', [])), n_cells),
        refined_ss=_as_per_cell_arrays(merged.get('refined_SS', []), n_cells),
        all_cs_spikes=_as_per_cell_arrays(merged.get('all_CS_spikes', []), n_cells),
        complex_bursts=_as_per_cell_entries(merged.get('complex_bursts_dicts', []), n_cells),
        plateaus=_as_per_cell_entries(merged.get('plateaus_dicts', []), n_cells),
        pos_nan_mask=pos_nan_mask,
        session_start_frames=session_start_frames,
        place_cell_mask=place_cell_mask,
        csplus_plc_mask=csplus_plc_mask,
        plc_categories=plc_categories,
        cb_in_pf_counts=cb_in_pf_counts,
        cs_peak_rates=cs_peak_rates,
        place_cell_source='spatial_analysis_full.pkl + refined final bad mask',
        snr_cache={},
    ))

cell_table = pd.DataFrame(cell_records)
print(f'Loaded refined payloads for {len(refined_animals)} animals and {len(cell_table)} cells.')
if unclassified_eligible:
    print(f'Warning: {len(unclassified_eligible)} eligible cells were not present in spatial_analysis_full.pkl.')
display(cell_table.groupby(['category', 'is_excluded']).size().rename('n_cells').reset_index())

Loaded refined payloads for 8 animals and 89 cells.


,category,is_excluded,n_cells
0,CS+ PLC,False,15
1,CS- PLC,False,10
2,Excluded,True,25
3,Non-PLC,False,39


## Build Category Cell Lists

The first three groups use refined spatial classification. The excluded group is defined by the final bad mask leaving zero good frames after the `min_good_minutes` rule.

In [3]:
def _ordered_keys_for_category(category):
    rows = cell_table.loc[cell_table['category'].eq(category), ['animal_id', 'cell_idx']]
    return [(str(row.animal_id), int(row.cell_idx)) for row in rows.itertuples(index=False)]


category_specs = [
    {
        'key': 'csplus',
        'label': 'CS+ PLCs',
        'cell_keys': _ordered_keys_for_category('CS+ PLC'),
        'html_name': 'refined_CSplus_PLCs_bad_frames_trace_viewer.html',
        'csv_name': 'refined_CSplus_PLCs_bad_frames_identity_map.csv',
    },
    {
        'key': 'csminus',
        'label': 'CS- PLCs',
        'cell_keys': _ordered_keys_for_category('CS- PLC'),
        'html_name': 'refined_CSminus_PLCs_bad_frames_trace_viewer.html',
        'csv_name': 'refined_CSminus_PLCs_bad_frames_identity_map.csv',
    },
    {
        'key': 'non_place',
        'label': 'Non-place cells',
        'cell_keys': _ordered_keys_for_category('Non-PLC'),
        'html_name': 'refined_non_place_cells_bad_frames_trace_viewer.html',
        'csv_name': 'refined_non_place_cells_bad_frames_identity_map.csv',
    },
    {
        'key': 'excluded',
        'label': 'Excluded cells',
        'cell_keys': _ordered_keys_for_category('Excluded'),
        'html_name': 'refined_excluded_cells_bad_frames_trace_viewer.html',
        'csv_name': 'refined_excluded_cells_bad_frames_identity_map.csv',
    },
]

summary_rows = []
for spec in category_specs:
    summary_rows.append({'group': spec['label'], 'n_cells': len(spec['cell_keys'])})
summary_df = pd.DataFrame(summary_rows)
display(summary_df)

excluded_by_mask = int(cell_table['is_excluded'].sum())
assert len(category_specs[-1]['cell_keys']) == excluded_by_mask, 'Excluded-cell list does not match final all-bad-mask count.'
print(f'Excluded-cell definition check passed: {excluded_by_mask} cells have final all-bad masks.')

,group,n_cells
0,CS+ PLCs,15
1,CS- PLCs,10
2,Non-place cells,39
3,Excluded cells,25


Excluded-cell definition check passed: 25 cells have final all-bad masks.


## Refined Viewer Helpers

The existing Plotly renderer is reused, but its SNR and bad-mask callbacks are replaced in this kernel so every dark gray interval is exactly the final refined bad mask.

In [4]:
fb.BAD_FILL = BAD_FILL_DARK


def refined_bad_mask_for_cell(animal, cell_idx, snr_threshold, min_good_minutes):
    masks = np.asarray(animal.merged.get('_refined_bad_masks'), dtype=bool)
    stats_list = animal.merged.get('_refined_mask_stats', [])
    if masks.ndim != 2 or int(cell_idx) >= masks.shape[0] or masks.shape[1] != int(animal.n_frames):
        bad_mask = np.ones(int(animal.n_frames), dtype=bool)
        stats = {
            'cell_idx': int(cell_idx),
            'n_frames_total': int(animal.n_frames),
            'n_removed_frames_total': int(animal.n_frames),
            'pct_removed_frames_total': 100.0,
            'eligible_cell': False,
            'removed_by_min_good_minutes': True,
        }
        return bad_mask, stats
    bad_mask = np.asarray(masks[int(cell_idx)], dtype=bool)
    stats = stats_list[int(cell_idx)] if isinstance(stats_list, list) and int(cell_idx) < len(stats_list) and isinstance(stats_list[int(cell_idx)], dict) else {}
    return bad_mask, dict(stats)


def refined_compute_snr_values(animal, cell_idx):
    return _get_cell_vector_from_source(animal.merged.get('SNR_interpolated', []), int(cell_idx), int(animal.n_frames), fill=np.nan)


fb.bad_mask_for_cell = refined_bad_mask_for_cell
fb.compute_snr_values = refined_compute_snr_values


def _display_order(cell_keys):
    wanted = {(str(animal_id), int(cell_idx)) for animal_id, cell_idx in cell_keys}
    ordered = []
    for animal in refined_animals:
        for cell_idx in range(int(animal.n_cells)):
            key = (str(animal.animal_id), int(cell_idx))
            if key in wanted:
                ordered.append(key)
    return ordered


def _write_identity_csv(cell_keys, csv_path):
    csv_path = Path(csv_path)
    csv_path.parent.mkdir(parents=True, exist_ok=True)
    lookup_df = cell_table.set_index(['animal_id', 'cell_idx'])
    fieldnames = [
        'row_number',
        'cell_identity',
        'animal_id',
        'cell_number',
        'cell_idx',
        'category',
        'included_minutes',
        'included_frames',
        'n_frames',
        'pct_bad_frames',
        'is_excluded',
        'snr_threshold',
        'min_good_minutes',
    ]
    with csv_path.open('w', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        for row_number, (animal_id, cell_idx) in enumerate(cell_keys, start=1):
            row = lookup_df.loc[(animal_id, cell_idx)]
            writer.writerow({
                'row_number': int(row_number),
                'cell_identity': f'{animal_id} | Cell {int(cell_idx) + 1} (idx {int(cell_idx)})',
                'animal_id': animal_id,
                'cell_number': int(cell_idx) + 1,
                'cell_idx': int(cell_idx),
                'category': str(row['category']),
                'included_minutes': f'{float(row["included_minutes"]):.6f}',
                'included_frames': int(row['included_frames']),
                'n_frames': int(row['n_frames']),
                'pct_bad_frames': f'{float(row["pct_bad_frames"]):.6f}',
                'is_excluded': bool(row['is_excluded']),
                'snr_threshold': f'{float(row["snr_threshold"]):g}',
                'min_good_minutes': f'{float(row["min_good_minutes"]):g}',
            })
    return csv_path


def export_refined_trace_viewer(spec, *, include_plotlyjs=True):
    ordered_keys = _display_order(spec['cell_keys'])
    key_filter = set(ordered_keys)
    y_label_lookup = {key: str(i) for i, key in enumerate(ordered_keys, start=1)}
    output_html = output_dir / spec['html_name']
    output_csv = output_dir / spec['csv_name']
    max_duration_s = max((animal.n_frames / float(animal.frame_rate) for animal in refined_animals), default=0.0)
    windows = fb._segment_windows(max_duration_s, SEGMENT_DURATION_S)

    segments = []
    first_fig = None
    first_stats = []
    for window in windows:
        fig, stats_rows = fb.build_pooled_place_cells_figure(
            refined_animals,
            snr_threshold=float(config.analysis.snr_threshold),
            min_good_minutes=float(config.analysis.min_good_minutes),
            trace_max_points=None,
            time_window_s=window,
            category_filter=None,
            optional_layers_visible=True,
            bad_epochs_visible=True,
            row_height_px=fb.DEFAULT_STACKED_ROW_HEIGHT_PX,
            show_legend=False,
            y_label_mode='number',
            y_label_lookup=y_label_lookup,
            cell_selection='all',
            cell_key_filter=key_filter,
        )
        fig.update_layout(
            title=f"{spec['label']}: refined traces with final bad frames shaded",
            yaxis_title='Cell number',
        )
        if first_fig is None:
            first_fig = fig
            first_stats = stats_rows
        start_s, end_s = window
        segments.append({
            'label': f'{start_s:g}-{end_s:g} s',
            'start_label': fb._format_segment_start_minute(start_s),
            'start_s': float(start_s),
            'end_s': float(end_s),
            'figure': fig.to_plotly_json(),
        })

    fb._write_segmented_pooled_html(
        output_path=output_html,
        segments=segments,
        include_plotlyjs=include_plotlyjs,
        title=f"{spec['label']} refined bad-frame trace viewer",
    )
    csv_path = _write_identity_csv(ordered_keys, output_csv)
    return {
        'group': spec['label'],
        'html_path': output_html,
        'csv_path': csv_path,
        'n_cells': len(ordered_keys),
        'n_segments': len(windows),
        'first_segment_rows': len(first_stats),
    }

print('Viewer helpers ready. Bad-frame fill:', fb.BAD_FILL)

Viewer helpers ready. Bad-frame fill: rgba(45, 45, 45, 0.70)


## Export Four HTML Viewers

This writes exactly four HTML files plus four identity CSVs under `figures/CKII_pooled/refined_trace_viewers/`.

In [5]:
export_results = []
for spec in category_specs:
    result = export_refined_trace_viewer(spec, include_plotlyjs=True)
    export_results.append(result)
    print(
        f"{result['group']}: saved {result['html_path']} "
        f"({result['n_cells']} cells, {result['n_segments']} segment(s)); CSV: {result['csv_path']}"
    )

export_df = pd.DataFrame(export_results)
display(export_df)

expected_html = {spec['html_name'] for spec in category_specs}
actual_html = {p.name for p in output_dir.glob('*.html') if p.name in expected_html}
assert actual_html == expected_html, f'Missing expected HTML outputs: {sorted(expected_html - actual_html)}'
print(f'Confirmed {len(actual_html)} expected HTML files in {output_dir}.')

CS+ PLCs: saved /Users/qixinyang/Documents/AdamLab/miniVI_Renana_Pipeline/miniVI_PlaceCell_analysis_V4/figures/CKII_pooled/refined_trace_viewers/refined_CSplus_PLCs_bad_frames_trace_viewer.html (15 cells, 10 segment(s)); CSV: /Users/qixinyang/Documents/AdamLab/miniVI_Renana_Pipeline/miniVI_PlaceCell_analysis_V4/figures/CKII_pooled/refined_trace_viewers/refined_CSplus_PLCs_bad_frames_identity_map.csv
CS- PLCs: saved /Users/qixinyang/Documents/AdamLab/miniVI_Renana_Pipeline/miniVI_PlaceCell_analysis_V4/figures/CKII_pooled/refined_trace_viewers/refined_CSminus_PLCs_bad_frames_trace_viewer.html (10 cells, 10 segment(s)); CSV: /Users/qixinyang/Documents/AdamLab/miniVI_Renana_Pipeline/miniVI_PlaceCell_analysis_V4/figures/CKII_pooled/refined_trace_viewers/refined_CSminus_PLCs_bad_frames_identity_map.csv
Non-place cells: saved /Users/qixinyang/Documents/AdamLab/miniVI_Renana_Pipeline/miniVI_PlaceCell_analysis_V4/figures/CKII_pooled/refined_trace_viewers/refined_non_place_cells_bad_frames_trace

,group,html_path,csv_path,n_cells,n_segments,first_segment_rows
0,CS+ PLCs,/Users/qixinyang/Documents/AdamLab/miniVI_Rena...,/Users/qixinyang/Documents/AdamLab/miniVI_Rena...,15,10,15
1,CS- PLCs,/Users/qixinyang/Documents/AdamLab/miniVI_Rena...,/Users/qixinyang/Documents/AdamLab/miniVI_Rena...,10,10,10
2,Non-place cells,/Users/qixinyang/Documents/AdamLab/miniVI_Rena...,/Users/qixinyang/Documents/AdamLab/miniVI_Rena...,39,10,39
3,Excluded cells,/Users/qixinyang/Documents/AdamLab/miniVI_Rena...,/Users/qixinyang/Documents/AdamLab/miniVI_Rena...,25,10,25


Confirmed 4 expected HTML files in /Users/qixinyang/Documents/AdamLab/miniVI_Renana_Pipeline/miniVI_PlaceCell_analysis_V4/figures/CKII_pooled/refined_trace_viewers.


## Spot Check: CKII_pAce46_PR_20260222 Cell 7

This confirms that a known cell's dark gray intervals come from the same final bad mask used by `_compute_bad_masks`.

In [ ]:
spot_animal_id = 'CKII_pAce46_PR_20260222'
spot_cell_idx = 6  # Cell7, zero-based index 6
spot_animal = next(a for a in refined_animals if a.animal_id == spot_animal_id)
spot_bad_mask, spot_stats = refined_bad_mask_for_cell(
    spot_animal,
    spot_cell_idx,
    config.analysis.snr_threshold,
    config.analysis.min_good_minutes,
)
spot_snr = refined_compute_snr_values(spot_animal, spot_cell_idx)
spot_good = ~spot_bad_mask
first_low = np.flatnonzero((~np.isfinite(spot_snr)) | (spot_snr < float(config.analysis.snr_threshold)))
spot_summary = {
    'animal_id': spot_animal_id,
    'cell_number': spot_cell_idx + 1,
    'included_frames': int(np.sum(spot_good)),
    'included_minutes': float(np.sum(spot_good)) / float(spot_animal.frame_rate) / 60.0,
    'pct_bad_frames': float(100.0 * np.sum(spot_bad_mask) / spot_animal.n_frames),
    'first_snr_below_threshold_min': float(first_low[0] / spot_animal.frame_rate / 60.0) if first_low.size else np.nan,
    'last_included_timepoint_min': float(np.flatnonzero(spot_good)[-1] / spot_animal.frame_rate / 60.0) if np.any(spot_good) else np.nan,
    'removed_by_min_good_minutes': bool(spot_stats.get('removed_by_min_good_minutes', False)),
}
display(pd.DataFrame([spot_summary]))